# Customer Support Tone - Ridge Regression

Train on `train.csv`, evaluate on the supplied `validation.csv`, and create a `test.csv` submission. Change the hyperparameters and rerun from the configuration cell onward.

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import f1_score, mean_absolute_error
from sklearn.pipeline import Pipeline

DATA_DIR = Path.cwd()
if not (DATA_DIR / 'train.csv').exists():
    DATA_DIR = DATA_DIR / 'customersupporttone'

# Hyperparameters.
RIDGE_ALPHA = 10.0
WORD_NGRAM_RANGE = (1, 2)
CHAR_NGRAM_RANGE = (3, 5)
MIN_DF = 1
MAX_FEATURES = None

train_df = pd.read_csv(DATA_DIR / 'train.csv')
dev_df = pd.read_csv(DATA_DIR / 'validation.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')

def combine_text(frame):
    return (frame['conversation'].fillna('').astype(str) +
            ' [LAST_CUSTOMER] ' + frame['last_customer_message'].fillna('').astype(str))

X_train = pd.DataFrame({'text': combine_text(train_df)})
X_dev = pd.DataFrame({'text': combine_text(dev_df)})
X_test = pd.DataFrame({'text': combine_text(test_df)})
y_train = train_df['label']
y_dev = dev_df['label']
print(f'Train: {len(train_df)} | validation: {len(dev_df)} | test: {len(test_df)}')

Train: 302 | validation: 103 | test: 101


## Validation metrics

In [4]:
features = ColumnTransformer([
    ('word_features', TfidfVectorizer(ngram_range=WORD_NGRAM_RANGE, min_df=MIN_DF, max_features=MAX_FEATURES, sublinear_tf=True), 'text'),
    ('char_features', TfidfVectorizer(analyzer='char', ngram_range=CHAR_NGRAM_RANGE, min_df=MIN_DF, max_features=MAX_FEATURES, sublinear_tf=True), 'text'),
])
model = Pipeline([('features', features), ('regressor', Ridge(alpha=RIDGE_ALPHA))])
model.fit(X_train, y_train)
dev_predictions = np.clip(model.predict(X_dev), 1, 5)
dev_class_predictions = np.rint(dev_predictions).astype(int)
dev_mae = mean_absolute_error(y_dev, dev_predictions)
dev_macro_f1 = f1_score(y_dev, dev_class_predictions, average='macro')
print(f'Validation MAE: {dev_mae:.4f}')
print(f'Validation macro-F1: {dev_macro_f1:.4f}')

Validation MAE: 0.9765
Validation macro-F1: 0.0974


## Final fit and submission

In [5]:
model.fit(pd.concat([X_train, X_dev], ignore_index=True), pd.concat([y_train, y_dev], ignore_index=True))
test_predictions = np.clip(model.predict(X_test), 1, 5)
submission = pd.DataFrame({'id': test_df['id'], 'label': test_predictions})
submission_path = DATA_DIR / 'submission_ridge.csv'
submission.to_csv(submission_path, index=False)
print(f'Saved {len(submission)} predictions to {submission_path}')
if 'label' in test_df.columns:
    print(f'Test macro-F1: {f1_score(test_df["label"], np.rint(test_predictions).astype(int), average="macro"):.4f}')
else:
    print('Test macro-F1 unavailable: test.csv has no ground-truth label column.')
submission.head()

Saved 101 predictions to c:\Users\bradm\Documents\GitHub\ai-masters-workspace\ARI-510-Machine-Learning\Phase 2b\customersupporttone\submission_ridge.csv
Test macro-F1 unavailable: test.csv has no ground-truth label column.


,id,label
0,18972,3.135324
1,49147,3.492675
2,86244,3.390117
3,98577,3.342088
4,99913,3.255593
